In [ ]:

# 1. --- UNIT PARSING & LOADING ---
unit_map = {'f': 1e-15, 'p': 1e-12, 'n': 1e-9, 'u': 1e-6, 'm': 1e-3, 'k': 1e3, 'M': 1e6, 'G': 1e9}

def parse_units(value):
    if pd.isna(value) or str(value).strip().lower() == "error" or str(value).strip() == "":
        return np.nan
    val_str = str(value).strip()
    match = re.search(r'([0-9\.-]+)([a-zA-Z]*)', val_str)
    if match:
        num = float(match.group(1))
        multiplier = unit_map.get(match.group(2), 1.0)
        return num * multiplier
    return np.nan

# --- File Paths ---
fmax_files = {
    'ind': "ind_at_9.5Ghz_MEMO.csv",
    'cc': "CC_min_PrimeSim_default_justCC_Measurements_history_1_20260410_16_56_40.57.csv",
    'var': "Var_char_Fmax_newvar_Tese_Varactor_characterization_tb2_Measurements_history_1_20260612_14_35_30.37.csv"
}

fmin_files = {
    'ind': "ind_at_6Ghz_MEMOMEMO_PrimeSim_default_Measurements_history_1_20260602_17_18_30.19.csv",
    'cc': "CC_max_PrimeSim_default_justCC_Measurements_history_1_20260410_16_41_15.9.csv",
    'var': "Var_char_Fmin_newvar_Tese_Varactor_characterization_tb2_Measurements_history_1_20260612_14_15_30.35.csv"
}

# --- Data Processing Function ---
def process_data(files):
    if not all(os.path.exists(f) for f in files.values()):
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    # 1. Inductor Data
    df_i_raw = pd.read_csv(files['ind'])
    # Handle L_min:ac vs L_max:ac automatically
    l_col = 'L_min:ac' if 'L_min:ac' in df_i_raw.columns else 'L_max:ac'
    ind_data = []
    for _, row in df_i_raw.iterrows():
        c_str = str(row['Corner'])
        key = 'TT' if '_TT' in c_str else 'bcQ' if '_bcQ' in c_str else 'wcQ' if '_wcQ' in c_str else None
        if key:
            ind_data.append({
                'Corner': key, 
                'Temp': row['Corner:Variable:temp'], 
                'L': parse_units(row[l_col]), 
                'Q_ind': float(row['Q:ac'])
            })
    df_ind = pd.DataFrame(ind_data)

    # 2. Cap Bank Data
    df_c_raw = pd.read_csv(files['cc'])
    cc_col = 'Cap_measured_min:ac' if 'Cap_measured_min:ac' in df_c_raw.columns else 'Cap_measured_max:ac'
    df_cc = pd.DataFrame({
        'Corner': df_c_raw['Corner'],
        'Temp': df_c_raw['Sweep:Variable:temp'],
        'CC': df_c_raw[cc_col].apply(parse_units)
    })

    # 3. Varactor Data
    df_v_raw = pd.read_csv(files['var'])
    df_var = pd.DataFrame({
        'Corner': df_v_raw['Corner'],
        'Temp': df_v_raw['Sweep:Variable:temp'].apply(parse_units),
        # Multiplied by 2 here as requested
        'Cvar': df_v_raw['Cap_measuredAt1Mhz_M7:ac'].apply(parse_units) * 2,
        'Q_var': df_v_raw['Q_factorAt1Mhz:ac'].apply(parse_units)
    }).dropna()

    return df_ind, df_cc, df_var

# --- Configuration ---
combinations = {
    'Wcq (Worst)': ('wcQ', 'wcQ125', 'SS125'),
    'Bcq (Best)': ('bcQ', 'bcQ-40', 'FF-40'),
    'Typical': ('TT', 'Typ25', 'TT25')
}
colors = {'Wcq (Worst)': 'red', 'Bcq (Best)': 'blue', 'Typical': 'green'}
C_parasitic = 114.50e-15 # Parsed from 114.50 fF

# --- 2. PROCESSING & PLOTTING ---
fig, (ax_f, ax_q) = plt.subplots(2, 1, figsize=(10, 14))
plt.subplots_adjust(hspace=0.3)
summary_points = []
target_temps = [-40, 25, 125]

def analyze_configuration(files, label_suffix, linestyle, C_switches):
    df_ind, df_cc, df_var = process_data(files)
    if df_ind.empty: return

    for label, (l_key, cc_key, v_key) in combinations.items():
        i_sub = df_ind[df_ind['Corner'] == l_key].sort_values('Temp')
        cc_sub = df_cc[df_cc['Corner'] == cc_key].sort_values('Temp')
        v_sub = df_var[df_var['Corner'] == v_key].sort_values('Temp')
        
        if i_sub.empty or cc_sub.empty or v_sub.empty: continue

        f_L = interp1d(i_sub['Temp'], i_sub['L'], fill_value="extrapolate")
        f_Qi = interp1d(i_sub['Temp'], i_sub['Q_ind'], fill_value="extrapolate")
        f_CC = interp1d(cc_sub['Temp'], cc_sub['CC'], fill_value="extrapolate")
        f_Cvar = interp1d(v_sub['Temp'], v_sub['Cvar'], fill_value="extrapolate")
        f_Qv = interp1d(v_sub['Temp'], v_sub['Q_var'], fill_value="extrapolate")

        t_range = np.linspace(-40, 125, 100)
        L_v, Qi_v, CC_v, Cv_v, Qv_v = f_L(t_range), f_Qi(t_range), f_CC(t_range), f_Cvar(t_range), f_Qv(t_range)
        
        # Calculate Total Capacitance array including new C_switches
        C_tot_v = CC_v + Cv_v + C_parasitic + C_switches
        
        freq = (1 / (2 * np.pi * np.sqrt(L_v * C_tot_v))) / 1e9
        q_tot = (Qi_v * Qv_v) / (Qi_v + Qv_v)
        
        ax_f.plot(t_range, freq, label=f"{label} ({label_suffix})", color=colors[label], linestyle=linestyle, linewidth=2)
        ax_q.plot(t_range, q_tot, label=f"{label} ({label_suffix})", color=colors[label], linestyle=linestyle, linewidth=2)

        # Collect points
        res = {'Label': f"{label} ({label_suffix})"}
        for t in target_temps:
            Ct = f_CC(t) + f_Cvar(t) + C_parasitic + C_switches
            res[f'f_{t}'] = (1 / (2 * np.pi * np.sqrt(f_L(t) * Ct))) / 1e9
            res[f'q_{t}'] = (f_Qi(t) * f_Qv(t)) / (f_Qi(t) + f_Qv(t))
        
        f_sw = freq
        delta_freq_ghz = np.max(f_sw) - np.min(f_sw)
        res['f_delta_ghz'] = delta_freq_ghz
        res['f_delta_mhz'] = delta_freq_ghz * 1000
        res['f_var_pct'] = (delta_freq_ghz / np.mean(f_sw)) * 100
        
        # --- Calculate L and C variations over temperature ---
        res['L_var_pct'] = (np.max(L_v) - np.min(L_v)) / np.mean(L_v) * 100
        res['C_var_pct'] = (np.max(C_tot_v) - np.min(C_tot_v)) / np.mean(C_tot_v) * 100
        
        summary_points.append(res)

# Appending the extra C_switches calculated to shift the center frequency
C_sw_min_freq = 1612.48e-15 # From earlier: Switch capacitance for 6.0 GHz fmin point
C_sw_max_freq = 200.30e-15  # From earlier: Switch capacitance for 9.5 GHz fmax point

analyze_configuration(fmin_files, "fmin", "-", C_sw_min_freq)
analyze_configuration(fmax_files, "fmax", "--", C_sw_max_freq)

# Formatting
ax_f.set_title("Tank Resonance Frequency Overlap", fontweight='bold')
ax_f.set_ylabel("Frequency (GHz)")
ax_f.grid(True, linestyle='--', alpha=0.6)
ax_f.legend()

ax_q.set_title(r"Tank Total Quality Factor ($Q_{total} = \frac{Q_L Q_{Cvar}}{Q_L + Q_{Cvar}}$)", fontweight='bold')
ax_q.set_ylabel("Total Q")
ax_q.set_xlabel("Temperature (°C)")
ax_q.grid(True, linestyle='--', alpha=0.6)
ax_q.legend()

plt.tight_layout()
plt.show()

# --- 3. PRINT SUMMARY ---
print("\n" + "="*145)
print(f"{'Corner Combination':<25} | {'Temp':<6} | {'Freq (GHz)':<12} | {'Total Q':<10} | {'Delta Freq':<12} | {'%Dev Freq':<10} | {'L Var %':<10} | {'C Var %':<10}")
print("-" * 145)
for res in summary_points:
    for i, t in enumerate(target_temps):
        l_str = res['Label'] if i == 1 else ""
        d_str = f"{res['f_delta_mhz']:>7.2f} MHz" if i == 1 else ""
        freq_val = res[f'f_{t}']
        freq_dev_pct = (res['f_delta_ghz'] / freq_val) * 100 if freq_val else np.nan
        
        # Only print L and C variance on the middle line
        l_var_str = f"{res['L_var_pct']:>8.3f}%" if i == 1 else ""
        c_var_str = f"{res['C_var_pct']:>8.3f}%" if i == 1 else ""

        print(f"{l_str:<25} | {t:>4}°C | {freq_val:>10.4f}   | {res[f'q_{t}']:>8.2f}   | {d_str:<12} | {freq_dev_pct:>9.3f}% | {l_var_str:<10} | {c_var_str:<10}")
    print(f"{'':<25} | %Var | {res['f_var_pct']:>9.3f}%   | {'':<10} | {'':<12} | {'':<10} | {'':<10} | {'':<10}")
    print("-" * 145)